# Esta parte do código se refere à pipeline da camada GOLD em BATCH para testes antes de subir ao AWS

In [235]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~
# Instalando as dependências
# ~~~~~~~~~~~~~~~~~~~~~~~~~~

# pyarrow para salvar em PARQUET

!pip install pyarrow colorama tabulate --quiet


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: C:\Users\carol\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [236]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~
# Importações
# ~~~~~~~~~~~~~~~~~~~~~~~~~~
import logging
from pathlib import Path
from datetime import datetime

import pandas as pd

In [237]:
# ~~~~~~~~~~~~~~~
# CONFIGURAÇÕES
# ~~~~~~~~~~~~~~~
from pathlib import Path
from datetime import datetime

DATA_SILVER = Path("silver")
DATA_GOLD = Path("gold")

DATA_GOLD.mkdir(parents=True, exist_ok=True)

PROCESSAMENTO = datetime.now()

In [238]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~
# CONFIGURAÇÃO DOS LOGS
# ~~~~~~~~~~~~~~~~~~~~~~~~~~
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(message)s"
)

log = logging.getLogger(__name__)

In [239]:
# ~~~~~~~~~~~~~~~
# LOG INICIAL
# ~~~~~~~~~~~~~~~

log.info("~" * 35)
log.info("INICIANDO ETL DA CAMADA GOLD")
log.info("~" * 35)

2026-07-12 23:37:41,398 | INFO     | ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
2026-07-12 23:37:41,399 | INFO     | INICIANDO ETL DA CAMADA GOLD
2026-07-12 23:37:41,400 | INFO     | ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~


In [240]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# LENDO ARQUIVOS DA CAMADA SILVER
"""
    Lê um arquivo Parquet da camada SILVER.

    Args:
        tabela (str): Nome da tabela.
    """
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
def ler_silver(tabela):

    caminho = DATA_SILVER / f"{tabela}.parquet"

    log.info(f"Lendo Silver: {caminho}")

    return pd.read_parquet(caminho)

In [ ]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# GOLD - RANKING UF
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

def gold_ranking_uf():

    log.info("Construindo Gold: Ranking UF")

    df = ler_silver("uf")

    # removendo o Total, pois só há uma única informação
    df = df[
        df["rede"] != "Total (Federal, Estadual, Municipal e Privada)"
    ].copy()

    # ordena
    df = df.sort_values(
        by=["ano", "rede", "taxa_alfabetizacao"],
        ascending=[True, True, False]
    )

    # ranking por ano e rede
    df["ranking"] = (
        df.groupby(
            ["ano", "rede"]
        )["taxa_alfabetizacao"]
        .rank(
            method="dense",
            ascending=False
        )
        .astype(int)
    )

    df["_gold_processed_at"] = datetime.now()
    
    df = df[
        [
            "ano",
            "sigla_uf",
            "sigla_uf_nome",
            "rede",
            "taxa_alfabetizacao",
            "ranking",
            "_gold_processed_at"
        ]
    ]

    log.info("Ranking UF criado")

    return df

In [ ]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# GOLD - RANKING MUNICÍPIOS
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

def gold_ranking_municipio():

    log.info("Construindo Gold: Ranking Municípios")

    df = ler_silver("municipio")

    # Ordena por ano, rede e taxa
    df = df.sort_values(
        by=["ano", "rede", "taxa_alfabetizacao"],
        ascending=[True, True, False]
    )

    # Cria ranking por ano e rede
    df["ranking"] = (
        df.groupby(
            ["ano", "rede"]
        )["taxa_alfabetizacao"]
        .rank(
            method="dense",
            ascending=False
        )
        .astype(int)
    )

    df["_gold_processed_at"] = datetime.now()

    # Mantém somente as colunas importantes
    df = df[
        [
            "ano",
            "id_municipio",
            "id_municipio_nome",
            "rede",
            "taxa_alfabetizacao",
            "ranking",
            "_gold_processed_at"
        ]
    ]

    log.info("Ranking de Municípios criado")

    return df

In [ ]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# GOLD - EVOLUÇÃO UF
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
def gold_evolucao_uf():

    log.info("Construindo Gold: Evolução UF")

    df = ler_silver("uf")

    df = df[
        [
            "ano",
            "sigla_uf",
            "sigla_uf_nome",
            "rede",
            "taxa_alfabetizacao",
            "media_portugues"
        ]
    ].copy()

    df["_gold_processed_at"] = datetime.now()

    df = df.sort_values(
        by=["sigla_uf", "rede", "ano"]
    )

    log.info("Gold Evolução UF criada")

    return df

In [ ]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# GOLD - RESUMO POR REDE
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

def gold_resumo_rede():

    log.info("Construindo Gold: Resumo por Rede")

    df = ler_silver("uf")
    df = df[
        df["rede"] != "Total (Federal, Estadual, Municipal e Privada)"
    ]

    df_gold = (        
        df.groupby(["ano", "rede"])
          .agg(
              media_taxa_alfabetizacao=(
                  "taxa_alfabetizacao",
                  "mean"
              ),
              media_portugues=(
                  "media_portugues",
                  "mean"
              ),
              quantidade_ufs=(
                  "sigla_uf",
                  "nunique"
              )
          )
          .reset_index()
    )
    
    df_gold["media_taxa_alfabetizacao"] = (
    df_gold["media_taxa_alfabetizacao"].round(2)
    )

    df_gold["media_portugues"] = (
        df_gold["media_portugues"].round(2)
    )

    df_gold["_gold_processed_at"] = datetime.now()

    log.info("Resumo por Rede criado")

    return df_gold

In [ ]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# FUNÇÃO: EXTRAIR A META DO PRÓPRIO ANO DA LINHA
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

def _extrair_meta_do_ano(df):
    """
    Args:
        df (pandas.DataFrame): Tabela de meta (uf ou município), contendo
            a coluna `ano` e as colunas `meta_alfabetizacao_2024..2030`.

    Returns:
        pandas.Series: Valor da meta definida para o próprio `ano` da
            linha (NaN se não houver meta definida para aquele ano, ex.:
            anos anteriores a 2024).
    """

    colunas_meta = {
        ano: f"meta_alfabetizacao_{ano}"
        for ano in range(2024, 2031)
        if f"meta_alfabetizacao_{ano}" in df.columns
    }

    meta_do_ano = pd.Series(np.nan, index=df.index, dtype="float64")

    for ano, coluna in colunas_meta.items():
        mascara = df["ano"] == ano
        meta_do_ano.loc[mascara] = df.loc[mascara, coluna]

    return meta_do_ano

In [ ]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# GOLD - COMPARAÇÃO META VS. RESULTADO (UF)
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

def gold_comparacao_meta_uf():

    log.info("Construindo Gold: Comparação Meta vs. Resultado - UF")

    df = ler_silver("meta_alfabetizacao_uf")

    # removendo o Total, mesmo critério usado no ranking/evolução de UF
    df = df[
        df["rede"] != "Total (Federal, Estadual, Municipal e Privada)"
    ].copy()

    df["meta_do_ano"] = _extrair_meta_do_ano(df)

    # só faz sentido comparar quando existe meta definida para aquele ano
    df = df[df["meta_do_ano"].notna()].copy()

    df["diferenca_pp"] = (
        df["taxa_alfabetizacao"] - df["meta_do_ano"]
    ).round(2)

    df["atingiu_meta"] = df["diferenca_pp"] >= 0

    df["_gold_processed_at"] = datetime.now()

    df = df[
        [
            "ano",
            "sigla_uf",
            "sigla_uf_nome",
            "rede",
            "taxa_alfabetizacao",
            "meta_do_ano",
            "diferenca_pp",
            "atingiu_meta",
            "_gold_processed_at"
        ]
    ]

    df = df.sort_values(
        by=["sigla_uf", "rede", "ano"]
    )

    log.info("Gold Comparação Meta vs. Resultado (UF) criada")

    return df

In [ ]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# GOLD - COMPARAÇÃO META VS. RESULTADO (MUNICÍPIO)
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

def gold_comparacao_meta_municipio():

    log.info("Construindo Gold: Comparação Meta vs. Resultado - Município")

    df = ler_silver("meta_alfabetizacao_municipio")

    df = df[
        df["rede"] != "Total (Federal, Estadual, Municipal e Privada)"
    ].copy()

    df["meta_do_ano"] = _extrair_meta_do_ano(df)

    df = df[df["meta_do_ano"].notna()].copy()

    df["diferenca_pp"] = (
        df["taxa_alfabetizacao"] - df["meta_do_ano"]
    ).round(2)

    df["atingiu_meta"] = df["diferenca_pp"] >= 0

    df["_gold_processed_at"] = datetime.now()

    df = df[
        [
            "ano",
            "id_municipio",
            "id_municipio_nome",
            "rede",
            "taxa_alfabetizacao",
            "meta_do_ano",
            "diferenca_pp",
            "atingiu_meta",
            "nivel_alfabetizacao",
            "_gold_processed_at"
        ]
    ]

    df = df.sort_values(
        by=["id_municipio", "rede", "ano"]
    )

    log.info("Gold Comparação Meta vs. Resultado (Município) criada")

    return df

In [ ]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# REGRAS DE DATA QUALITY
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
CHECKS = {

    "gold_ranking_uf": [

        {
            "tipo": "min_count",
            "valor": 100,
            "critico": True
        },

        {
            "tipo": "not_null",
            "coluna": "sigla_uf",
            "critico": True
        },

        {
            "tipo": "not_null",
            "coluna": "ranking",
            "critico": True
        },

        {
            "tipo": "range",
            "coluna": "taxa_alfabetizacao",
            "valor": (0,100),
            "critico": True
        }

    ],
    "ranking_municipio": [

        {
            "tipo": "min_count",
            "valor": 1000,
            "critico": True
        },

        {
            "tipo": "not_null",
            "coluna": "id_municipio",
            "critico": True
        },

        {
            "tipo": "not_null",
            "coluna": "ranking",
            "critico": True
        },

        {
            "tipo": "range",
            "coluna": "taxa_alfabetizacao",
            "valor": (0, 100),
            "critico": True
        }
    ],
    "evolucao_uf": [

        {
            "tipo": "min_count",
            "valor": 100,
            "critico": True
        },

        {
            "tipo": "not_null",
            "coluna": "sigla_uf",
            "critico": True
        },

        {
            "tipo": "not_null",
            "coluna": "ano",
            "critico": True
        },

        {
            "tipo": "range",
            "coluna": "taxa_alfabetizacao",
            "valor": (0, 100),
            "critico": True
        }

    ],
    "resumo_rede": [

    {
        "tipo": "min_count",
        "valor": 7,
        "critico": True
    },

    {
        "tipo": "not_null",
        "coluna": "ano",
        "critico": True
    },

    {
        "tipo": "not_null",
        "coluna": "rede",
        "critico": True
    },

    {
        "tipo": "range",
        "coluna": "media_taxa_alfabetizacao",
        "valor": (0, 100),
        "critico": True
    }

],

    "comparacao_meta_uf": [

        {
            # valor conservador: como só existem metas para 2024-2030,
            # o volume real depende de quantos desses anos já têm
            # avaliação (ano) registrada na base. Ajustar após a
            # primeira execução com dados reais.
            "tipo": "min_count",
            "valor": 10,
            "critico": True
        },

        {
            "tipo": "not_null",
            "coluna": "sigla_uf",
            "critico": True
        },

        {
            "tipo": "not_null",
            "coluna": "meta_do_ano",
            "critico": True
        },

        {
            "tipo": "range",
            "coluna": "taxa_alfabetizacao",
            "valor": (0, 100),
            "critico": True
        },

        {
            "tipo": "range",
            "coluna": "meta_do_ano",
            "valor": (0, 100),
            "critico": True
        }

    ],

    "comparacao_meta_municipio": [

        {
            "tipo": "min_count",
            "valor": 50,
            "critico": True
        },

        {
            "tipo": "not_null",
            "coluna": "id_municipio",
            "critico": True
        },

        {
            "tipo": "not_null",
            "coluna": "meta_do_ano",
            "critico": True
        },

        {
            "tipo": "range",
            "coluna": "taxa_alfabetizacao",
            "valor": (0, 100),
            "critico": True
        },

        {
            "tipo": "range",
            "coluna": "meta_do_ano",
            "valor": (0, 100),
            "critico": True
        }

    ]

}

In [ ]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# FUNÇÃO DE QUALIDADE DA CAMADA GOLD
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

def checar_qualidade(df, checks):

    log.info("Iniciando verificações de qualidade")

    for check in checks:

        if check["tipo"] == "min_count":

            assert len(df) >= check["valor"], \
                f"Quantidade mínima não atendida ({len(df)} registros)."

        elif check["tipo"] == "not_null":

            coluna = check["coluna"]

            assert df[coluna].isnull().sum() == 0, \
                f"Existem valores nulos na coluna '{coluna}'."

        elif check["tipo"] == "range":

            coluna = check["coluna"]
            minimo, maximo = check["valor"]

            assert (
                df[coluna].between(minimo, maximo).all()
            ), f"Valores fora do intervalo na coluna '{coluna}'."

    log.info("Checks de qualidade concluídos com sucesso!")

In [247]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# SALVA UM DATAFRAME NA CAMADA GOLD EM FORMATO PARQUET
"""
    Args:
        df (pandas.DataFrame): DataFrame tratado.
"""
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
def salvar_gold(df, nome):

    caminho = DATA_GOLD / f"{nome}.parquet"

    df.to_parquet(
        caminho,
        index=False
    )

    log.info(f"Camada GOLD salva em {caminho}")

    return caminho

In [ ]:
# ~~~~~~~~~~~~~~~~~~~~~~~~
# EXECUÇÃO DA CAMADA GOLD
# ~~~~~~~~~~~~~~~~~~~~~~~~
def executar_gold():

    log.info("~" * 35)
    log.info("INICIANDO CAMADA GOLD")
    log.info("~" * 35)

    # ~~~~~~~~~~~
    # Ranking UF
    # ~~~~~~~~~~~

    df_gold = gold_ranking_uf()
    log.info("Checando qualidade de Ranking UF")

    checar_qualidade(
        df_gold,
        CHECKS["gold_ranking_uf"]
    )

    salvar_gold(
        df_gold,
        "ranking_uf"
    )

    # ~~~~~~~~~~~~~~~~~~
    # Ranking Municípios
    # ~~~~~~~~~~~~~~~~~~
    log.info("Checando qualidade de Ranking Municipio")

    df_gold = gold_ranking_municipio()

    checar_qualidade(
        df_gold,
        CHECKS["ranking_municipio"]
    )

    salvar_gold(
        df_gold,
        "ranking_municipio"
    )

    # ~~~~~~~~~~~~~~~~~~
    # Evolução UF
    # ~~~~~~~~~~~~~~~~~~
    log.info("Checando qualidade de Evolução UF")

    df_gold = gold_evolucao_uf()

    checar_qualidade(
        df_gold,
        CHECKS["evolucao_uf"]
    )

    salvar_gold(
        df_gold,
        "evolucao_uf"
    )

    # ~~~~~~~~~~~~~~~~~~
    # Resumo por Rede
    # ~~~~~~~~~~~~~~~~~~
    log.info("Checando qualidade de Resumo por Rede")

    df_gold = gold_resumo_rede()

    checar_qualidade(
        df_gold,
        CHECKS["resumo_rede"]
    )
    salvar_gold(
        df_gold,
        "resumo_rede"
    )

    # ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    # Comparação Meta vs. Resultado - UF
    # ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    log.info("Checando qualidade de Comparação Meta vs. Resultado (UF)")

    df_gold = gold_comparacao_meta_uf()

    checar_qualidade(
        df_gold,
        CHECKS["comparacao_meta_uf"]
    )

    salvar_gold(
        df_gold,
        "comparacao_meta_uf"
    )

    # ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    # Comparação Meta vs. Resultado - Município
    # ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    log.info("Checando qualidade de Comparação Meta vs. Resultado (Município)")

    df_gold = gold_comparacao_meta_municipio()

    checar_qualidade(
        df_gold,
        CHECKS["comparacao_meta_municipio"]
    )

    salvar_gold(
        df_gold,
        "comparacao_meta_municipio"
    )

    log.info("Camada Gold concluída!")

In [249]:
executar_gold()

2026-07-12 23:37:41,521 | INFO     | ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
2026-07-12 23:37:41,521 | INFO     | INICIANDO CAMADA GOLD
2026-07-12 23:37:41,522 | INFO     | ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
2026-07-12 23:37:41,523 | INFO     | Construindo Gold: Ranking UF
2026-07-12 23:37:41,523 | INFO     | Lendo Silver: silver\uf.parquet
2026-07-12 23:37:41,533 | INFO     | Ranking UF criado
2026-07-12 23:37:41,534 | INFO     | Checando qualidade de Ranking UF
2026-07-12 23:37:41,534 | INFO     | Iniciando verificações de qualidade
2026-07-12 23:37:41,536 | INFO     | Checks de qualidade concluídos com sucesso!
2026-07-12 23:37:41,541 | INFO     | Camada GOLD salva em gold\ranking_uf.parquet
2026-07-12 23:37:41,542 | INFO     | Checando qualidade de Ranking Municipio
2026-07-12 23:37:41,543 | INFO     | Construindo Gold: Ranking Municípios
2026-07-12 23:37:41,544 | INFO     | Lendo Silver: silver\municipio.parquet
2026-07-12 23:37:41,575 | INFO     | Ranking de Municípios criado
2026-0